In [1]:
print("hi")

hi


In [39]:
%pwd

'c:\\Users\\Pratham\\OneDrive\\Desktop\\chatbot-mokshfit\\chatbot'

In [3]:
#since I am right now in research folder(-\\research). I have to come back and enter into Data folder

In [4]:
import os 
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\Pratham\\OneDrive\\Desktop\\chatbot-mokshfit\\chatbot'

1. DATA EXTRACTION OF THE JOSN FILE.

In [7]:
from langchain_community.document_loaders import DirectoryLoader, JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [46]:
import json
from langchain.schema import Document

def load_json_file(file_path):
    documents = []

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for item in data:

        # ---- Minimal Metadata ----
        metadata = {
            "handle": item.get("Handle"),
            "title": item.get("Title"),
            "type": item.get("Type"),
            "price": item.get("Variant Price"),
            "published": item.get("Published")
        }

        # ---- Rich Page Content ----
        page_content = f"""
        Product Title: {item.get('Title')}
        Product Handle: {item.get('Handle')}
        Product Type: {item.get('Type')}
        Vendor: {item.get('Vendor')}
        Color: {item.get('Color')}
        Size: {item.get('Size')}
        Fabric Type: {item.get('Fabric_Type')}
        GSM: {item.get('GSM')}
        Fit Type: {item.get('Fit_Type')}
        Care Instructions: {item.get('Care_Instructions')}
        Tags: {item.get('Tags')}
        Image URL: {item.get('Image Src')}
        Image Position: {item.get('Image Position')}
        Image Alt Text: {item.get('Image Alt Text')}
        """

        documents.append(
            Document(
                metadata=metadata,
                page_content=page_content.strip()
            )
        )

    return documents

In [47]:
extracted_docs = load_json_file("data/mokshfit.json")

print("Total Documents:", len(extracted_docs))
print(extracted_docs[0])

Total Documents: 509
page_content='Product Title: Unisex Oversized Shirt Moksfit
        Product Handle: unisex-oversized-shirt-moksfit
        Product Type: T-Shirt
        Vendor: Mokshfit
        Color: White
        Size: S
        Fabric Type: 100% cotton
        GSM: 240
        Fit Type: Oversized fit with half sleeves and straight hem
        Care Instructions: Wash inside-out in cold water, dry on low heat. Flip it inside out before ironing.
        Tags: Streetwear, Summer, Structured, Oversized, Premium Cotton, Unisex, Breathable
        Image URL: https://cdn.shopify.com/s/files/1/0811/4588/8993/files/Left_Pocket_1_c_1.jpg?v=1770480555
        Image Position: 1
        Image Alt Text: Left_Pocket_1_c_1.jpg' metadata={'handle': 'unisex-oversized-shirt-moksfit', 'title': 'Unisex Oversized Shirt Moksfit', 'type': 'T-Shirt', 'price': '799', 'published': '2026-02-13T17:34:11+05:30'}


In [48]:
extracted_docs

[Document(metadata={'handle': 'unisex-oversized-shirt-moksfit', 'title': 'Unisex Oversized Shirt Moksfit', 'type': 'T-Shirt', 'price': '799', 'published': '2026-02-13T17:34:11+05:30'}, page_content='Product Title: Unisex Oversized Shirt Moksfit\n        Product Handle: unisex-oversized-shirt-moksfit\n        Product Type: T-Shirt\n        Vendor: Mokshfit\n        Color: White\n        Size: S\n        Fabric Type: 100% cotton\n        GSM: 240\n        Fit Type: Oversized fit with half sleeves and straight hem\n        Care Instructions: Wash inside-out in cold water, dry on low heat. Flip it inside out before ironing.\n        Tags: Streetwear, Summer, Structured, Oversized, Premium Cotton, Unisex, Breathable\n        Image URL: https://cdn.shopify.com/s/files/1/0811/4588/8993/files/Left_Pocket_1_c_1.jpg?v=1770480555\n        Image Position: 1\n        Image Alt Text: Left_Pocket_1_c_1.jpg'),
 Document(metadata={'handle': 'unisex-oversized-shirt-moksfit', 'title': 'Unisex Oversized S

In [67]:
# Since the page content is not cleaned like \n next page , etc...
import re

def clean_product_text(text):

    # --- FIX WORDS BROKEN BY HYPHEN + NEWLINE ---
    text = re.sub(r'(\w+)-\s*[\r\n]+\s*(\w+)', r'\1\2', text)

    # --- PROTECT PARAGRAPHS ---
    text = text.replace('\n\n', '[[PARA]]')

    # --- REMOVE SINGLE NEWLINES/TABS ---
    text = text.replace('\n', '.').replace('\r', ' ').replace('\t', ' ')

    # --- RESTORE PARAGRAPHS ---
    text = text.replace('[[PARA]]', '\n\n')

    # --- REMOVE EXTRA SPACES ---
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [68]:
def clean_documents(documents):

    for doc in documents:
        doc.page_content = clean_product_text(doc.page_content)

    return documents

In [69]:
extracted_docs = load_json_file("data/mokshfit.json")

# Clean the documents
cleaned_docs = clean_documents(extracted_docs)

print("Total Documents:", len(cleaned_docs))
print(cleaned_docs[0]) # checking if above funvtion work or not.

Total Documents: 509
page_content='Product Title: Unisex Oversized Shirt Moksfit. Product Handle: unisex-oversized-shirt-moksfit. Product Type: T-Shirt. Vendor: Mokshfit. Color: White. Size: S. Fabric Type: 100% cotton. GSM: 240. Fit Type: Oversized fit with half sleeves and straight hem. Care Instructions: Wash inside-out in cold water, dry on low heat. Flip it inside out before ironing.. Tags: Streetwear, Summer, Structured, Oversized, Premium Cotton, Unisex, Breathable. Image URL: https://cdn.shopify.com/s/files/1/0811/4588/8993/files/Left_Pocket_1_c_1.jpg?v=1770480555. Image Position: 1. Image Alt Text: Left_Pocket_1_c_1.jpg' metadata={'handle': 'unisex-oversized-shirt-moksfit', 'title': 'Unisex Oversized Shirt Moksfit', 'type': 'T-Shirt', 'price': '799', 'published': '2026-02-13T17:34:11+05:30'}


In [70]:
cleaned_docs

[Document(metadata={'handle': 'unisex-oversized-shirt-moksfit', 'title': 'Unisex Oversized Shirt Moksfit', 'type': 'T-Shirt', 'price': '799', 'published': '2026-02-13T17:34:11+05:30'}, page_content='Product Title: Unisex Oversized Shirt Moksfit. Product Handle: unisex-oversized-shirt-moksfit. Product Type: T-Shirt. Vendor: Mokshfit. Color: White. Size: S. Fabric Type: 100% cotton. GSM: 240. Fit Type: Oversized fit with half sleeves and straight hem. Care Instructions: Wash inside-out in cold water, dry on low heat. Flip it inside out before ironing.. Tags: Streetwear, Summer, Structured, Oversized, Premium Cotton, Unisex, Breathable. Image URL: https://cdn.shopify.com/s/files/1/0811/4588/8993/files/Left_Pocket_1_c_1.jpg?v=1770480555. Image Position: 1. Image Alt Text: Left_Pocket_1_c_1.jpg'),
 Document(metadata={'handle': 'unisex-oversized-shirt-moksfit', 'title': 'Unisex Oversized Shirt Moksfit', 'type': 'T-Shirt', 'price': '799', 'published': '2026-02-13T17:34:11+05:31'}, page_conten

2. CHUNKING

In [71]:
# For product catalog we don't require chunking coz already we have one product in one documnent. So we can directly use cleaned_docs for vectorization and embedding.

3. EMBEDDING MODELS

In [75]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

In [77]:
embeddings

HuggingFaceEmbeddings(model_name='BAAI/bge-base-en-v1.5', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [83]:
vector = embeddings.embed_query("Hello world")
vector

[0.010724288411438465,
 0.05578271672129631,
 0.027084119617938995,
 0.003040965646505356,
 0.03033558279275894,
 0.020480435341596603,
 0.03132845461368561,
 0.04103910177946091,
 -0.025208787992596626,
 -0.0572720430791378,
 -0.003961740527302027,
 -0.004360872320830822,
 -0.06811681389808655,
 0.0195290707051754,
 0.016956325620412827,
 0.028180846944451332,
 0.0315973274409771,
 0.0007251513889059424,
 0.015515688806772232,
 0.037916459143161774,
 -0.05291704088449478,
 0.00934568326920271,
 0.03269621357321739,
 0.01581241376698017,
 -0.0061231376603245735,
 -0.00772967329248786,
 0.0018662899965420365,
 0.04321456328034401,
 -0.09220433235168457,
 -0.005244026426225901,
 0.023606574162840843,
 0.006433933973312378,
 0.019542457535862923,
 -0.03940853476524353,
 0.0038774011190980673,
 0.023421457037329674,
 0.0010544854449108243,
 0.0031184267718344927,
 -0.015605452470481396,
 -0.03244800120592117,
 -0.014721883460879326,
 -0.006371771916747093,
 -0.0014695621794089675,
 0.01682

In [84]:
len(vector)

768

4. VECTOR DB

In [86]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [88]:
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")
HUGGINGFACE_API_KEY=os.getenv("HUGGINGFACE_API_KEY")


os.environ["PINECONE_API_KEY"]=PINECONE_API_KEY
os.environ["HUGGINGFACE_API_KEY"]=HUGGINGFACE_API_KEY

In [89]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

#Authenticate Pinecone
pc = Pinecone(api_key=pinecone_api_key)

In [90]:
pc

In [93]:
from pinecone import ServerlessSpec

index_name = "mokshfit"

if not pc.has_index(index_name):
        pc.create_index(
            name = index_name,
            dimension = 768, # Dimension of the embeddings
            metric = "cosine", # Cosine Similarity
            spec = ServerlessSpec( cloud = "aws" , region="us-east-1")
        )
index = pc.Index(index_name)

In [125]:
doc_ids = []
for i, doc in enumerate(cleaned_docs, start=1):
    # Pull the title from metadata (fallback to handle if missing)
    # E.g., "Unisex Oversized Shirt Moksfit"
    title = doc.metadata.get("title", doc.metadata.get("handle", "Unknown"))
    
    # Replace any hyphens with spaces (just in case it pulled the handle)
    clean_title = title.replace("-", " ")
    
    # Get the first letter of every word and make it uppercase
    # "Unisex Oversized Shirt Moksfit" -> "UOSM"
    initials = "".join([word[0].upper() for word in clean_title.split() if word])
    
    # Combine the acronym with the index number
    unique_id = f"{initials}_{i}"
    doc_ids.append(unique_id)

In [126]:
doc_ids # Check the first 20 generated document IDs

['UOSM_1',
 'UOSM_2',
 'UOSM_3',
 'UOSM_4',
 'UOSM_5',
 'UOSM_6',
 'UOSM_7',
 'UOSM_8',
 'UOSM_9',
 'UOSM_10',
 'UOSM_11',
 'UOSM_12',
 'UOSM_13',
 'UOSM_14',
 'UOSM_15',
 'UOSM_16',
 'UOSM_17',
 'UOSM_18',
 'UOSM_19',
 'UOSM_20',
 'UOSM_21',
 'UOSM_22',
 'UOSM_23',
 'UOSM_24',
 'UOSM_25',
 'USCTS_26',
 'USCTS_27',
 'USCTS_28',
 'USCTS_29',
 'USCTS_30',
 'USCTS_31',
 'USCTS_32',
 'USCTS_33',
 'USCTS_34',
 'USCTS_35',
 'USCTS_36',
 'USCTS_37',
 'USCTS_38',
 'UOHHHM_39',
 'UOHHHM_40',
 'UOHHHM_41',
 'UOHHHM_42',
 'UOHHHM_43',
 'UOHHHM_44',
 'UOHHHM_45',
 'UOHHHM_46',
 'UOHHHM_47',
 'UOHHHM_48',
 'UOHHHM_49',
 'UOHHHM_50',
 'UOHHHM_51',
 'UOHHHM_52',
 'UOHHHM_53',
 'UOHHHM_54',
 'UOHHHM_55',
 'UOHHHM_56',
 'UOHHHM_57',
 'UOHHHM_58',
 'UOHHHM_59',
 'UOHHHM_60',
 'UOHHHM_61',
 'UOHHHM_62',
 'UOHHHM_63',
 'UOCTS_64',
 'UOCTS_65',
 'UOCTS_66',
 'UOCTS_67',
 'UOCTS_68',
 'UOCTS_69',
 'UOCTS_70',
 'UOCTS_71',
 'UOCTS_72',
 'UOCTS_73',
 'UOCTS_74',
 'UOCTS_75',
 'UOCTS_76',
 'UOCTS_77',
 'UOCTS_

In [127]:
from langchain_pinecone import PineconeVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")

batch_size = 100 # pinecone allows up to 100 vectors per upsert
index_name = "mokshfit"

print(f"Starting upload for {len(cleaned_docs)} documents...")
print(f"Example ID created: {doc_ids[0]}") # Lets you see what it looks like before uploading!

for i in range(0, len(cleaned_docs), batch_size):
    batch_docs = cleaned_docs[i : i + batch_size]
    batch_ids = doc_ids[i : i + batch_size]
    
    # Passing 'ids=batch_ids' forces Pinecone to OVERWRITE instead of DUPLICATE
    PineconeVectorStore.from_documents(
        documents=batch_docs,
        embedding=embeddings,
        index_name=index_name,
        ids=batch_ids 
    )
    print(f"Processed batch covering documents {i+1} to {i + len(batch_docs)}")

# 4. Connect to the Vector Store for future querying
docsearch = PineconeVectorStore(index_name=index_name, embedding=embeddings)

print("\n✅ Success! Your Mokshfit data is now in Pinecone.")
print("🔁 If you run this block again, it will safely overwrite existing vectors using IDs like UOSM_1, UOSM_2, etc.")

Starting upload for 509 documents...
Example ID created: UOSM_1
Processed batch covering documents 1 to 100
Processed batch covering documents 101 to 200
Processed batch covering documents 201 to 300
Processed batch covering documents 301 to 400
Processed batch covering documents 401 to 500
Processed batch covering documents 501 to 509

✅ Success! Your Mokshfit data is now in Pinecone.
🔁 If you run this block again, it will safely overwrite existing vectors using IDs like UOSM_1, UOSM_2, etc.


In [128]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [129]:
retrieved_docs = retriever.invoke("Is there har har mahadev hoodies available?")
retrieved_docs

[Document(id='UOHHHM_55', metadata={'handle': 'unisex-oversized-hoodie-har-har-mahadev', 'human_id': '', 'price': '1699', 'published': '2026-02-13T17:34:10+05:46', 'title': 'Unisex Oversized Hoodie Har Har Mahadev', 'type': 'Hoodie'}, page_content='Product Title: Unisex Oversized Hoodie Har Har Mahadev. Product Handle: unisex-oversized-hoodie-har-har-mahadev. Product Type: Hoodie. Vendor: Mokshfit. Color: Maroon. Size: M. Fabric Type: 100% cotton. GSM: 400. Fit Type: Unisex oversized fit with dropped shoulders. Care Instructions: Wash inside-out in cold water, dry on low heat. Flip it inside out before ironing.. Tags: Heavyweight, Winter Wear, Religious, Spiritual, Graphic, Oversized, Premium, Cozy. Image URL: https://cdn.shopify.com/s/files/1/0811/4588/8993/files/Front_1_c_25_77dfe320-86be-4774-984d-5a8e4f2f42dd.jpg?v=1770480595. Image Position: 4. Image Alt Text: Front_1_c_25.jpg'),
 Document(id='UOHHHM_58', metadata={'handle': 'unisex-oversized-hoodie-har-har-mahadev', 'human_id': '

In [136]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":25})

In [137]:
retrieved_docs = retriever.invoke("Is there har har mahadev hoodies available?")
retrieved_docs

[Document(id='UOHHHM_55', metadata={'handle': 'unisex-oversized-hoodie-har-har-mahadev', 'human_id': '', 'price': '1699', 'published': '2026-02-13T17:34:10+05:46', 'title': 'Unisex Oversized Hoodie Har Har Mahadev', 'type': 'Hoodie'}, page_content='Product Title: Unisex Oversized Hoodie Har Har Mahadev. Product Handle: unisex-oversized-hoodie-har-har-mahadev. Product Type: Hoodie. Vendor: Mokshfit. Color: Maroon. Size: M. Fabric Type: 100% cotton. GSM: 400. Fit Type: Unisex oversized fit with dropped shoulders. Care Instructions: Wash inside-out in cold water, dry on low heat. Flip it inside out before ironing.. Tags: Heavyweight, Winter Wear, Religious, Spiritual, Graphic, Oversized, Premium, Cozy. Image URL: https://cdn.shopify.com/s/files/1/0811/4588/8993/files/Front_1_c_25_77dfe320-86be-4774-984d-5a8e4f2f42dd.jpg?v=1770480595. Image Position: 4. Image Alt Text: Front_1_c_25.jpg'),
 Document(id='UOHHHM_58', metadata={'handle': 'unisex-oversized-hoodie-har-har-mahadev', 'human_id': '

In [139]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":30})

In [140]:
retrieved_docs = retriever.invoke("Is there har har mahadev hoodies available?")
retrieved_docs

[Document(id='UOHHHM_55', metadata={'handle': 'unisex-oversized-hoodie-har-har-mahadev', 'human_id': '', 'price': '1699', 'published': '2026-02-13T17:34:10+05:46', 'title': 'Unisex Oversized Hoodie Har Har Mahadev', 'type': 'Hoodie'}, page_content='Product Title: Unisex Oversized Hoodie Har Har Mahadev. Product Handle: unisex-oversized-hoodie-har-har-mahadev. Product Type: Hoodie. Vendor: Mokshfit. Color: Maroon. Size: M. Fabric Type: 100% cotton. GSM: 400. Fit Type: Unisex oversized fit with dropped shoulders. Care Instructions: Wash inside-out in cold water, dry on low heat. Flip it inside out before ironing.. Tags: Heavyweight, Winter Wear, Religious, Spiritual, Graphic, Oversized, Premium, Cozy. Image URL: https://cdn.shopify.com/s/files/1/0811/4588/8993/files/Front_1_c_25_77dfe320-86be-4774-984d-5a8e4f2f42dd.jpg?v=1770480595. Image Position: 4. Image Alt Text: Front_1_c_25.jpg'),
 Document(id='UOHHHM_58', metadata={'handle': 'unisex-oversized-hoodie-har-har-mahadev', 'human_id': '

5. LLM

In [218]:
import os
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

# 1. Load the key from your .env file
load_dotenv()
my_key = os.getenv("HUGGINGFACE_API_KEY_2")  # Use the second key if you want to test with a different one

# 2. CRITICAL FIX: Set the exact environment variables the library demands!
os.environ["HF_TOKEN"] = my_key
os.environ["HUGGINGFACEHUB_API_TOKEN"] = my_key

print("Loading Qwen 2.5 via native Hugging Face...")

# 3. Initialize the endpoint (it will now automatically grab the token from the environment)
endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    max_new_tokens=512,
    temperature=0.2
)

# 4. Wrap it in the Chat class
chat_model = ChatHuggingFace(llm=endpoint)

print("✅ Hugging Face Chat Model Loaded Successfully!")

Loading Qwen 2.5 via native Hugging Face...
✅ Hugging Face Chat Model Loaded Successfully!


In [219]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

In [220]:
prompt_template = """You are a premium streetwear sales expert for the brand MOKSHFIT. 
Use the following pieces of retrieved product context to answer the user's question. 
If you don't know the answer or the product isn't in the context, just say that you don't have that information right now. Do not make up prices, colors, or sizes. Keep your tone helpful, energetic, and modern.

Context: {context}

Question: {question}

Helpful Answer:"""

PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

In [221]:
PROMPT

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are a premium streetwear sales expert for the brand MOKSHFIT. \nUse the following pieces of retrieved product context to answer the user's question. \nIf you don't know the answer or the product isn't in the context, just say that you don't have that information right now. Do not make up prices, colors, or sizes. Keep your tone helpful, energetic, and modern.\n\nContext: {context}\n\nQuestion: {question}\n\nHelpful Answer:")

6. CHAINING

In [222]:
qa_chain = RetrievalQA.from_chain_type(
    llm=chat_model, # <--- Using native ChatHuggingFace
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}
)
print("✅ Chatbot RAG Chain is Ready!")

✅ Chatbot RAG Chain is Ready!


In [223]:
question = "Do you have the Har Har Mahadev hoodie? What is the GSM and price?"

print(f"👤 User: {question}\n")

# Get the answer
response = qa_chain.invoke({"query": question})

print(f"🤖 Mokshfit Bot:\n{response['result']}")

👤 User: Do you have the Har Har Mahadev hoodie? What is the GSM and price?

🤖 Mokshfit Bot:
Yes, we do have the Har Har Mahadev hoodie! It's a premium, unisex oversized hoodie with a dropped shoulder fit. The GSM (Grams per Square Meter) for this hoodie is 400, which makes it a bit heavier and more substantial compared to lighter hoodies.

Unfortunately, the exact price isn't listed here, but you can find it on our website or in our app. If you'd like, I can help you check the current price for you. Would you like to proceed with that?


In [224]:
question = "Do you have any winter wear cloths? What is the GSM and price?"

print(f"👤 User: {question}\n")

# Get the answer
response = qa_chain.invoke({"query": question})

print(f"🤖 Mokshfit Bot:\n{response['result']}")

👤 User: Do you have any winter wear cloths? What is the GSM and price?

🤖 Mokshfit Bot:
Absolutely! Our MOKSHFIT Unisex Hoodies are perfect for winter wear. They are made from 100% cotton, brushed fleece with a GSM of 300, which ensures they are soft, warm, and comfortable. While I don't have specific price information at hand, you can check the current pricing on our website or in our retail stores. For more details, you can visit our product page for the Unisex Hoodie Mokshfit. If you need further assistance or have any other questions, feel free to ask!


In [227]:
question = "Do you have something in 400 or greather than that GSM cloths?"

print(f"👤 User: {question}\n")

# Get the answer
response = qa_chain.invoke({"query": question})

print(f"🤖 Mokshfit Bot:\n{response['result']}")

👤 User: Do you have something in 400 or greather than that GSM cloths?

🤖 Mokshfit Bot:
I don't have products available with a GSM (Grams per Square Meter) of 400 or greater. The heaviest option we offer is 240 GSM, which you can find in various colors and sizes including Maroon, Black, Petrol Blue, and Navy Blue. If you're looking for something even heavier, you might want to check out other brands or collections that specialize in ultra-heavyweight tees. Thanks for your interest in MOKSHFIT!


In [228]:
question = "What is best seller?"

print(f"👤 User: {question}\n")

# Get the answer
response = qa_chain.invoke({"query": question})

print(f"🤖 Mokshfit Bot:\n{response['result']}")

👤 User: What is best seller?

🤖 Mokshfit Bot:
Based on the information provided, it's not explicitly clear which specific product is the best seller. However, the products that seem to be available in multiple sizes and colors, which might indicate higher demand, are the MOKSHFIT SPECIALs PRINTED T-Shirts in Beige and Baby Blue. These colors are offered in sizes XS, S, M, L, and XXL. Given the variety of sizes, it's likely that these colors have broader appeal and might be more popular among customers.

If you're looking for a specific best-seller, I recommend checking the sales data or customer preferences directly from the Mokshfit platform. Would you like more details on any of these colors or sizes?


In [229]:
question = "Anything in White and Small size?"

print(f"👤 User: {question}\n")

# Get the answer
response = qa_chain.invoke({"query": question})

print(f"🤖 Mokshfit Bot:\n{response['result']}")

👤 User: Anything in White and Small size?

🤖 Mokshfit Bot:
Yes, we do have a White Unisex Classic Crew T-Shirt in Small size (S). It's made of 100% cotton with a GSM of 180, offering a perfect unisex regular fit. The care instructions are to wash inside-out in cold water, dry on low heat, and flip it inside out before ironing. This t-shirt is a great Everyday Essential, Lightweight, and Budget-friendly option. You can check it out [here](https://cdn.shopify.com/s/files/1/0811/4588/8993/files/Back_2_c_1_b2b28612-3ce4-4be2-a5e3-d8d1f1906560.jpg?v=1770479643).


In [232]:
question = "Who is Elon musk?"

print(f"👤 User: {question}\n")

# Get the answer
response = qa_chain.invoke({"query": question})

print(f"🤖 Mokshfit Bot:\n{response['result']}")

👤 User: Who is Elon musk?

🤖 Mokshfit Bot:
I don't have that information right now. Based on the product context provided, it seems we specialize in MOKSHFIT's streetwear collection, particularly their Unisex Oversized Hoodies and Classic Crew T-Shirts. The context doesn't include any information about Elon Musk. If you're interested in learning about Elon Musk, you might want to check out reliable news sources or biographical websites.


In [233]:
question = "I want 50 T-Shirts customized priniting for my college event, is it possible?"

print(f"👤 User: {question}\n")

# Get the answer
response = qa_chain.invoke({"query": question})

print(f"🤖 Mokshfit Bot:\n{response['result']}")

👤 User: I want 50 T-Shirts customized priniting for my college event, is it possible?

🤖 Mokshfit Bot:
Absolutely! MOKSHFIT can definitely handle a bulk order of 50 T-Shirts for your college event. We specialize in custom printing and can work with you to design and print the perfect T-Shirts for your event. Our T-Shirts are made from 100% cotton, have a lightweight and perfect unisex fit, and are available in various colors and sizes to suit your needs.

To get started, you can provide us with the design you'd like to print and specify the color and size requirements. We'll take care of the rest, ensuring your T-Shirts are of the highest quality and perfect for your event.

Would you like to proceed with a design consultation or need more information on the customization process?


In [230]:
# After trial I understand that, I have to make prompt more structured and specific to get better answer.

In [231]:
# Trial testing on jupiter notebook is done now lets start with modular coding in scripts.

@app.route("/get", methods=["GET", "POST"])
def chat():
    msg = request.form["msg"]
    print(f"User Input: {msg}")
    
    response = qa_chain.invoke({"query": msg})
    result = response["result"]  # ← move this BEFORE jsonify
    
    print("Response : ", result)
    return jsonify({"answer": result})